# Paired webcam + lidar viewer

Interactive viewer for acquisition outputs (`<run>/images/*.jpg` + `<run>/lidar/*.npy`).
Set `RUN_DIR`, then run the cells. Scrub with the slider at the bottom.

In [ ]:
%matplotlib widget
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.image import imread
from ipywidgets import IntSlider, Dropdown, interact

# Import project-local helpers
REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO))
sys.path.insert(0, str(REPO / "examples"))

from examples.plot_paired import pair_files, render_lidar  # noqa: E402

In [ ]:
# >>> edit this to point at an acquisition run <<<
RUN_DIR = Path("/home/rfor10/lidar_data/20260421_demo_raw_backpack")

pairs = pair_files(RUN_DIR)
print(f"{len(pairs)} paired frames in {RUN_DIR}")
assert pairs, "no paired files found — check RUN_DIR"

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
im_cam = axes[0].imshow(np.zeros((2, 2, 3), dtype=np.uint8))
im_lid = axes[1].imshow(np.zeros((2, 2), dtype=np.uint8), cmap="jet")
for ax in axes:
    ax.axis("off")
title_cam = axes[0].set_title("")
title_lid = axes[1].set_title("")


def show(idx, view):
    img_path, lid_path = pairs[idx]
    img = imread(img_path)
    pts = np.load(lid_path)
    lidar_img, cmap = render_lidar(pts, view)

    im_cam.set_data(img)
    axes[0].set_xlim(0, img.shape[1]); axes[0].set_ylim(img.shape[0], 0)
    title_cam.set_text(f"webcam: {img_path.name}")

    im_lid.set_data(lidar_img)
    im_lid.set_cmap(cmap)
    im_lid.set_clim(lidar_img.min(), max(lidar_img.max(), 1))
    axes[1].set_xlim(0, lidar_img.shape[1]); axes[1].set_ylim(lidar_img.shape[0], 0)
    title_lid.set_text(f"lidar ({view}): {lid_path.name}  [{pts.shape[0]} pts]")

    fig.canvas.draw_idle()


interact(
    show,
    idx=IntSlider(min=0, max=len(pairs) - 1, step=1, value=0, description="frame"),
    view=Dropdown(options=["front", "pano", "bev"], value="front", description="view"),
);

## Save a montage of N sampled frames

In [ ]:
from examples.plot_paired import plot_pairs

stride = max(1, len(pairs) // 6)
plot_pairs(pairs[::stride][:6], view="front", save=None)